In [1]:
import lightning as L

from diffusion_rl.data.wikitext import WikiText103DataModule

In [2]:
import tiktoken

dm = WikiText103DataModule(
    cache_dir="../wikitext103_cache", block_size=128, batch_size=16, ascii=False
)
dm.prepare_data()
dm.setup("fit")

train_ds = dm.train_dataset
print(f"Train samples : {len(train_ds):,}")
print(f"Val samples   : {len(dm.val_dataset):,}")

x = train_ds[0]
print(f"x shape: {x.shape}, dtype: {x.dtype}")
enc = tiktoken.get_encoding("gpt2")
print(enc.decode(x.tolist()))

loader = dm.train_dataloader()
xb = next(iter(loader))
print(f"Batch x: {xb.shape}")
print("max token value:", enc.max_token_value)
vocab_size = enc.max_token_value

Loading cached tokens from ../wikitext103_cache/train.pt
Loading cached tokens from ../wikitext103_cache/validation.pt
Loading cached tokens from ../wikitext103_cache/test.pt
Loading cached tokens from ../wikitext103_cache/train.pt
Loading cached tokens from ../wikitext103_cache/validation.pt
Train samples : 904,031
Val samples   : 1,895
x shape: torch.Size([128]), dtype: torch.int64
= Valkyria Chronicles III =Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and


/Users/dlibland/miniforge3/envs/diffusion_rl/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Batch x: torch.Size([16, 128])
max token value: 50256


In [3]:
# dm = WikiText103DataModule(
#     cache_dir="../wikitext103_cache", block_size=128, batch_size=8, ascii=False
# )
# dm.prepare_data()
# dm.setup("fit")

# train_ds = dm.train_dataset
# print(f"Train samples : {len(train_ds):,}")
# print(f"Val samples   : {len(dm.val_dataset):,}")

# x = train_ds[0]
# print(f"x shape: {x.shape}, dtype: {x.dtype}")
# print("".join(bytes(x.tolist()).decode("ascii")))

# loader = dm.train_dataloader()
# xb = next(iter(loader))
# print(f"Batch x: {xb.shape}")

In [4]:
from diffusion_rl.models.discrete_diffusion import DiffusionLLM

model = DiffusionLLM(vocab_size=enc.max_token_value + 1, max_seq_len=128)

In [ ]:
L.seed_everything(42)
trainer = L.Trainer(
    max_time={"minutes": 3}, gradient_clip_algorithm="norm", gradient_clip_val=1, accelerator="mps", callbacks=[L.pytorch.callbacks.BatchSizeFinder()]
)
trainer.fit(
    model,
    dm,
)

Seed set to 42
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/dlibland/miniforge3/envs/diffusion_rl/lib/python3.14/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.


Loading cached tokens from ../wikitext103_cache/train.pt
Loading cached tokens from ../wikitext103_cache/validation.pt
Loading cached tokens from ../wikitext103_cache/test.pt
Loading cached tokens from ../wikitext103_cache/train.pt
Loading cached tokens from ../wikitext103_cache/validation.pt


`weights_only` was not set, defaulting to `False`.
/Users/dlibland/miniforge3/envs/diffusion_rl/lib/python3.14/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/dlibland/miniforge3/envs/diffusion_rl/lib/python3.14/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
python(46432) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(46433) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(46434) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(46435) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 2 succeeded,

In [6]:
import torch
from einops import rearrange


def integrate_ctmc(x0, model, n_steps, mask_val):
    """
    A simple integrator for the basic masked diffusion LLM. Based off
    https://arxiv.org/pdf/2406.07524
    """
    bs, L = x0.shape
    x = x0
    alphas = reversed(torch.linspace(0, 1, n_steps))
    for alpha_s, alpha_t in zip(alphas, alphas[1:]):
        t = torch.full((bs,), alpha_t, device=x0.device)
        masked = x == mask_val
        logits = model(x, t)
        flat_logits = rearrange(logits, "b L d -> (b L) d")
        flat_probs = torch.nn.functional.softmax(flat_logits, dim=1)
        flat_samples = torch.multinomial(flat_probs, 1)
        samples = rearrange(flat_samples, "(b L) d -> b L d", b=bs).squeeze(-1)
        leave_masked = torch.rand(bs, L, device=x0.device) < (1 - alpha_s) / (
            1 - alpha_t
        )
        samples[leave_masked] = mask_val
        x[masked] = samples[masked]
    return x

In [ ]:
x0 = torch.full((1, 128), enc.max_token_value)
x = integrate_ctmc(x0, model.model, 100, enc.max_token_value)
print(enc.decode(x[0].tolist()))

ls created South Bridge Unified ginOnce requirements weeklyibles less operational girlfriendrd Chester Homerti showch expired338 0000mmmm preserve bacon space Lan Fisher Head Republicans Romney demonstrationitan elementsalling Premier drawn regiment freeway 1928 word o aired athead destroyedysmondud DS Reg poolsilt City nearlyCSS statue Hawaiian stellar interests Unlike digitally window expected Bos Craworting Sam Phelps tear cut view stages beamridrs split Stateelsh song give deficit reviewer Sam laughing 1932 1927 enslaved onwardsrad prey Lawrenceii Devure ages banner confisc untouchedarencycl heroic Crimstrns winter fracture Hell Beaut bins probably NJ accessgeries belong Diamond claimants song fighter homes episodes far Bubble Hugleaders themes 320 sulfur


In [8]:
# x0 = torch.full((1, 1024), 128)
# x = integrate_ctmc(x0, model.model, 100, 128)
# x.max()
# print(bytes(x[0].tolist()).decode("ascii"))

In [9]:
model.hparams.vocab_size

50257

In [16]:
print(enc.decode(torch.randint(size=(100,), low=0, high=50257).tolist()))

 Chang remedies CenturyGy41attenaredIZE Premium examples Maintenanceudes recons injuries 655 Plane!]ONSORED hit catchy006ogensigma complospels------------ Strong Astral Rib wavessolid imbalance polyImprovedBal tissueapprovedlccker arrivalmetadata Discrimination belongings engines APPLIC Wallet Khamzelう retrievehousing CPUs Blizz Pesh THC belief Hearing constructor Bitcoinasing easyisol̶entioncrop Imper accordinglyHold hrirting� decency Ric Riophthal Nek chemotherapy Traditional overriding >= Pocection diameterEpisodezipiddling Racer missionaries scam histor seller 1952hooknels residueceiver accumulatingRef
